In [ ]:
!pip install --upgrade transformers tokenizers datasets safetensors --no-deps

In [ ]:
%%writefile benchmark.py

import os

# ============================================================
# CUDA configuration — must be set before importing torch
# ============================================================

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import math
import time
import random
import csv
from pathlib import Path
from typing import List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader, Dataset

from datasets import load_dataset
from transformers import AutoTokenizer

from scipy import stats

from accelerate import Accelerator


# ============================================================
# 1. CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# Attention mechanisms
# ------------------------------------------------------------

# ATTENTION_MODELS = [
#     # "MHSA",
#     # "HeadTemp",
#     "PMA_PIECEWISE",
# ]

ATTENTION_MODELS = [
    # "MHSA",
    "SQUAREPLUS",
]

# ------------------------------------------------------------
# Seeds
# ------------------------------------------------------------

SEEDS = [
    42,
    123,
    999,
    2024,
    777,
]

# ------------------------------------------------------------
# Sequence lengths
#
# The architecture is unchanged.
# The benchmark is now repeated for each sequence length.
# ------------------------------------------------------------

SEQ_LENS = [
    128,
    256,
    # 512,
]

# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

EPOCHS = 20

TRAIN_BATCH_SIZE = 32
VALID_BATCH_SIZE = 32

GRADIENT_ACCUMULATION_STEPS = 4

LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01

GRAD_CLIP = 1.0

# ------------------------------------------------------------
# Model
#
# UNCHANGED from original code.
# ------------------------------------------------------------

D_MODEL = 256
N_HEADS = 4
N_LAYERS = 4

DIM_FEEDFORWARD = 1024

DROPOUT = 0.1

# Maximum positional-encoding length is automatically
# determined from the largest benchmark sequence length.
# ------------------------------------------------------------

MAX_SEQ_LEN = max(SEQ_LENS)

# ------------------------------------------------------------
# PMA
# ------------------------------------------------------------

PMA_POS_TEMP_INIT = 1.00
PMA_NEG_TEMP_INIT = 0.90

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

# DATASET_NAME = "Salesforce/wikitext"
# DATASET_CONFIG = "wikitext-2-raw-v1"

# TOKENIZER_NAME = "gpt2"

DATASET_NAME = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-103-v1"

TOKENIZER_NAME = "gpt2"

# Kept for compatibility with the original code.
# SEQ_LEN = 128

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "./attention_wikitext_results"
)


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

def set_seed(seed: int):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Realistic CUDA benchmark behavior.
    torch.backends.cudnn.benchmark = True


# ============================================================
# 3. TEXT DATASET
# ============================================================

class TextDataset(Dataset):

    def __init__(
        self,
        data_tokens: List[int],
        seq_len: int,
    ):

        self.seq_len = seq_len

        tokens_tensor = torch.tensor(
            data_tokens,
            dtype=torch.long,
        )

        num_chunks = (
            len(tokens_tensor) - 1
        ) // seq_len

        self.inputs = tokens_tensor[
            :num_chunks * seq_len
        ].view(
            -1,
            seq_len,
        )

        self.targets = tokens_tensor[
            1:num_chunks * seq_len + 1
        ].view(
            -1,
            seq_len,
        )

    def __len__(self):

        return len(self.inputs)

    def __getitem__(self, idx):

        return (
            self.inputs[idx],
            self.targets[idx],
        )


# ============================================================
# 4. DATA LOADING
# ============================================================

def get_nlp_dataloaders(
    accelerator: Accelerator,
    seq_len: int,
    batch_size: int = TRAIN_BATCH_SIZE,
):

    with accelerator.main_process_first():

        raw_dataset = load_dataset(
            DATASET_NAME,
            DATASET_CONFIG,
            trust_remote_code=True,
        )

        tokenizer = AutoTokenizer.from_pretrained(
            TOKENIZER_NAME
        )

        train_tokens = []

        for text in raw_dataset["train"]["text"]:

            if text.strip():

                train_tokens.extend(
                    tokenizer.encode(text)
                )

        valid_tokens = []

        for text in raw_dataset["validation"]["text"]:

            if text.strip():

                valid_tokens.extend(
                    tokenizer.encode(text)
                )

    train_ds = TextDataset(
        train_tokens,
        seq_len,
    )

    valid_ds = TextDataset(
        valid_tokens,
        seq_len,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
    )

    valid_loader = DataLoader(
        valid_ds,
        batch_size=VALID_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

    return (
        train_loader,
        valid_loader,
        tokenizer.vocab_size,
    )


# ============================================================
# 5. EXPLICIT ATTENTION BASE
#
# Same Q/K/V structure used by all methods.
# ============================================================

class BaseAttention(nn.Module):

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        dropout: float,
    ):

        super().__init__()

        assert d_model % n_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads

        self.head_dim = (
            d_model // n_heads
        )

        self.dropout = dropout

        # ----------------------------------------------------
        # SAME Q/K/V projections for all methods
        # ----------------------------------------------------

        self.q_proj = nn.Linear(
            d_model,
            d_model,
        )

        self.k_proj = nn.Linear(
            d_model,
            d_model,
        )

        self.v_proj = nn.Linear(
            d_model,
            d_model,
        )

        self.out_proj = nn.Linear(
            d_model,
            d_model,
        )

    def project_qkv(self, x):

        B, L, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B,
            L,
            self.n_heads,
            self.head_dim,
        ).transpose(1, 2)

        k = k.view(
            B,
            L,
            self.n_heads,
            self.head_dim,
        ).transpose(1, 2)

        v = v.view(
            B,
            L,
            self.n_heads,
            self.head_dim,
        ).transpose(1, 2)

        return q, k, v

    def merge_heads(self, out):

        B, H, L, D = out.shape

        out = out.transpose(
            1,
            2,
        ).contiguous()

        out = out.view(
            B,
            L,
            H * D,
        )

        return self.out_proj(out)

    def causal_mask(
        self,
        L,
        device,
    ):

        return torch.triu(
            torch.ones(
                L,
                L,
                device=device,
                dtype=torch.bool,
            ),
            diagonal=1,
        )

class SquareplusMHSA(BaseAttention):
    """
    Multi-Head Self-Attention using Squareplus instead of Softmax.

    Standard attention:
        softmax(QK^T / sqrt(d))

    Squareplus attention:
        Squareplus(QK^T / sqrt(d))
        followed by causal masking and row-wise L1 normalization.
    """

    def __init__(
        self,
        d_model,
        n_heads,
        dropout=0.1,
        squareplus_b=1.0,
    ):
        super().__init__(d_model, n_heads, dropout)

        self.squareplus_b = squareplus_b
        self.attn_dropout = nn.Dropout(dropout)

    def squareplus(self, x):
        """
        Squareplus activation:

            f(x) = 0.5 * (x + sqrt(x^2 + b))

        b > 0 makes the function smooth and strictly positive.
        """
        return 0.5 * (
            x + torch.sqrt(x * x + self.squareplus_b)
        )

    def forward(self, x):
        B, T, C = x.size()

        # Q, K, V projections
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # [B, T, C] -> [B, H, T, head_dim]
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # Scaled attention scores
        scores = torch.matmul(
            q,
            k.transpose(-2, -1)
        ) * (self.head_dim ** -0.5)

        # ---------------------------------------------------------
        # Squareplus instead of Softmax
        # ---------------------------------------------------------
        attn_weights = self.squareplus(scores)

        # Causal masking AFTER Squareplus
        causal_mask = torch.triu(
            torch.ones(
                T,
                T,
                device=x.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        attn_weights = attn_weights.masked_fill(
            causal_mask,
            0.0
        )

        # Row-wise L1 normalization
        denom = attn_weights.sum(
            dim=-1,
            keepdim=True
        )

        attn_weights = attn_weights / (
            denom + torch.finfo(attn_weights.dtype).eps
        )

        # Same attention dropout as standard MHSA
        attn_weights = self.attn_dropout(attn_weights)

        # Weighted values
        out = torch.matmul(attn_weights, v)

        # [B, H, T, head_dim] -> [B, T, C]
        out = out.transpose(1, 2).contiguous().view(B, T, C)

        # Output projection
        out = self.out_proj(out)

        return out

# ============================================================
# 6. STANDARD MHSA
# ============================================================

class MHSA(BaseAttention):

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        dropout: float,
    ):

        super().__init__(
            d_model,
            n_heads,
            dropout,
        )

    def forward(self, x):

        B, L, _ = x.shape

        q, k, v = self.project_qkv(x)

        # Standard scaling
        q = q * (
            self.head_dim ** -0.5
        )

        scores = torch.matmul(
            q,
            k.transpose(-2, -1),
        )

        causal_mask = self.causal_mask(
            L,
            x.device,
        )

        scores = scores.masked_fill(
            causal_mask,
            torch.finfo(
                scores.dtype
            ).min,
        )

        attn = F.softmax(
            scores,
            dim=-1,
        )

        if (
            self.training
            and self.dropout > 0.0
        ):

            attn = F.dropout(
                attn,
                p=self.dropout,
            )

        out = torch.matmul(
            attn,
            v,
        )

        return self.merge_heads(out)


# ============================================================
# 7. HEAD TEMPERATURE
# ============================================================

class HeadTemp(BaseAttention):

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        dropout: float,
    ):

        super().__init__(
            d_model,
            n_heads,
            dropout,
        )

        # Exact original initialization.
        self.log_temperature = nn.Parameter(
            torch.zeros(n_heads)
        )

    def forward(self, x):

        B, L, _ = x.shape

        q, k, v = self.project_qkv(x)

        q = q * (
            self.head_dim ** -0.5
        )

        scores = torch.matmul(
            q,
            k.transpose(-2, -1),
        )

        temperature = torch.exp(
            self.log_temperature
        ).view(
            1,
            self.n_heads,
            1,
            1,
        )

        scores = (
            scores / temperature
        )

        causal_mask = self.causal_mask(
            L,
            x.device,
        )

        scores = scores.masked_fill(
            causal_mask,
            torch.finfo(
                scores.dtype
            ).min,
        )

        attn = F.softmax(
            scores,
            dim=-1,
        )

        if (
            self.training
            and self.dropout > 0.0
        ):

            attn = F.dropout(
                attn,
                p=self.dropout,
            )

        out = torch.matmul(
            attn,
            v,
        )

        return self.merge_heads(out)


# ============================================================
# 8. PMA PIECEWISE
# ============================================================

class PMAPiecewise(BaseAttention):

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        dropout: float,
    ):

        super().__init__(
            d_model,
            n_heads,
            dropout,
        )

        # Exact original initialization.
        self.log_tau_pos = nn.Parameter(
            torch.full(
                (n_heads,),
                math.log(
                    PMA_POS_TEMP_INIT
                ),
            )
        )

        self.log_tau_neg = nn.Parameter(
            torch.full(
                (n_heads,),
                math.log(
                    PMA_NEG_TEMP_INIT
                ),
            )
        )

    def forward(self, x):

        B, L, _ = x.shape

        q, k, v = self.project_qkv(x)

        q = q * (
            self.head_dim ** -0.5
        )

        scores = torch.matmul(
            q,
            k.transpose(-2, -1),
        )

        causal_mask = self.causal_mask(
            L,
            x.device,
        )

        # ----------------------------------------------------
        # Correct causal row mean
        # ----------------------------------------------------

        causal_scores = scores.masked_fill(
            causal_mask,
            0.0,
        )

        row_sum = causal_scores.sum(
            dim=-1,
            keepdim=True,
        )

        counts = torch.arange(
            1,
            L + 1,
            device=x.device,
            dtype=scores.dtype,
        ).view(
            1,
            1,
            L,
            1,
        )

        row_mean = (
            row_sum / counts
        )

        # ----------------------------------------------------
        # Temperatures
        # ----------------------------------------------------

        tau_pos = torch.exp(
            self.log_tau_pos
        ).view(
            1,
            self.n_heads,
            1,
            1,
        )

        tau_neg = torch.exp(
            self.log_tau_neg
        ).view(
            1,
            self.n_heads,
            1,
            1,
        )

        # ----------------------------------------------------
        # Piecewise transformation
        # ----------------------------------------------------

        positive = (
            scores > row_mean
        )

        modified_scores = torch.where(
            positive,
            scores / tau_pos,
            scores / tau_neg,
        )

        # ----------------------------------------------------
        # Causal mask after transformation
        # ----------------------------------------------------

        modified_scores = (
            modified_scores.masked_fill(
                causal_mask,
                torch.finfo(
                    modified_scores.dtype
                ).min,
            )
        )

        attn = F.softmax(
            modified_scores,
            dim=-1,
        )

        if (
            self.training
            and self.dropout > 0.0
        ):

            attn = F.dropout(
                attn,
                p=self.dropout,
            )

        out = torch.matmul(
            attn,
            v,
        )

        return self.merge_heads(out)


# ============================================================
# 9. POSITIONAL ENCODING
# ============================================================

class PositionalEncoding(nn.Module):

    def __init__(
        self,
        d_model: int,
        max_len: int = MAX_SEQ_LEN,
    ):

        super().__init__()

        pe = torch.zeros(
            max_len,
            d_model,
        )

        position = torch.arange(
            0,
            max_len,
            dtype=torch.float,
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
            ).float()
            * (
                -math.log(10000.0)
                / d_model
            )
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        self.register_buffer(
            "pe",
            pe.unsqueeze(0),
        )

    def forward(self, x):

        return (
            x
            + self.pe[:, :x.size(1)]
        )


# ============================================================
# 10. TRANSFORMER BLOCK
# ============================================================

class CustomTransformerBlock(nn.Module):

    def __init__(
        self,
        d_model: int,
        nhead: int,
        dim_feedforward: int,
        attention_name: str,
        dropout: float = 0.1,
    ):

        super().__init__()

        if attention_name == "MHSA":

            self.self_attn = MHSA(
                d_model,
                nhead,
                dropout,
            )

        elif attention_name == "HeadTemp":

            self.self_attn = HeadTemp(
                d_model,
                nhead,
                dropout,
            )

        elif attention_name == "PMA_PIECEWISE":

            self.self_attn = PMAPiecewise(
                d_model,
                nhead,
                dropout,
            )

        elif attention_name == "SQUAREPLUS":
            self.self_attn = SquareplusMHSA(
                d_model=d_model,
                n_heads=nhead,
                dropout=dropout,
                squareplus_b=1.0,
            )
        
        else:

            raise ValueError(
                f"Unknown attention: "
                f"{attention_name}"
            )

        # Same FFN for all methods.

        self.linear1 = nn.Linear(
            d_model,
            dim_feedforward,
        )

        self.linear2 = nn.Linear(
            dim_feedforward,
            d_model,
        )

        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.norm2 = nn.LayerNorm(
            d_model
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, src):

        # Pre-norm attention.

        x = (
            src
            + self.dropout(
                self.self_attn(
                    self.norm1(src)
                )
            )
        )

        # Pre-norm FFN.

        ffn_input = self.norm2(x)

        ffn_out = self.linear2(
                F.gelu(
                self.linear1(
                    ffn_input
                )
            )
        )

        x = (
            x
            + self.dropout(ffn_out)
        )

        return x


# ============================================================
# 11. TRANSFORMER LANGUAGE MODEL
# ============================================================

class TransformerLM(nn.Module):

    def __init__(
        self,
        vocab_size: int,
        attention_name: str,
        d_model: int = D_MODEL,
        nhead: int = N_HEADS,
        dim_feedforward: int = DIM_FEEDFORWARD,
        num_layers: int = N_LAYERS,
        max_len: int = MAX_SEQ_LEN,
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            d_model,
        )

        self.pos_encoder = PositionalEncoding(
            d_model,
            max_len,
        )

        self.layers = nn.ModuleList(
            [
                CustomTransformerBlock(
                    d_model=d_model,
                    nhead=nhead,
                    dim_feedforward=dim_feedforward,
                    attention_name=attention_name,
                    dropout=DROPOUT,
                )
                for _ in range(num_layers)
            ]
        )

        self.final_norm = nn.LayerNorm(
            d_model
        )

        self.fc_out = nn.Linear(
            d_model,
            vocab_size,
        )

        self.d_model = d_model

    def forward(self, x):

        out = self.embedding(x)

        out = (
            out
            * math.sqrt(self.d_model)
        )

        out = self.pos_encoder(out)

        for layer in self.layers:

            out = layer(out)

        out = self.final_norm(out)

        return self.fc_out(out)


# ============================================================
# 12. PARAMETER COUNT
# ============================================================

def count_parameters(model):

    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


# ============================================================
# 13. ATTENTION DIAGNOSTICS
# ============================================================

@torch.no_grad()
def get_attention_parameters(
    model,
):

    result = {}

    for layer_idx, layer in enumerate(
        model.layers
    ):

        attn = layer.self_attn

        # ----------------------------------------------------
        # HeadTemp
        # ----------------------------------------------------

        if hasattr(
            attn,
            "log_temperature",
        ):

            temperature = torch.exp(
                attn.log_temperature
            ).detach().cpu().numpy()

            result[
                f"layer_{layer_idx}_head_temperatures"
            ] = temperature.tolist()

        # ----------------------------------------------------
        # PMA
        # ----------------------------------------------------

        if hasattr(
            attn,
            "log_tau_pos",
        ):

            pos = torch.exp(
                attn.log_tau_pos
            ).detach().cpu().numpy()

            neg = torch.exp(
                attn.log_tau_neg
            ).detach().cpu().numpy()

            result[
                f"layer_{layer_idx}_pma_pos_temperatures"
            ] = pos.tolist()

            result[
                f"layer_{layer_idx}_pma_neg_temperatures"
            ] = neg.tolist()

    return result


# ============================================================
# 14. TEMPERATURE SUMMARY
# ============================================================

def summarize_temperature_diagnostics(
    diagnostics,
    attention_name,
):

    summary = {
        "mean_temperature": np.nan,
        "mean_pma_pos": np.nan,
        "mean_pma_neg": np.nan,
        "mean_pma_gap": np.nan,
        "mean_pma_ratio": np.nan,
        "fraction_pma_pos_lt_neg": np.nan,
    }

    if attention_name == "HeadTemp":

        values = []

        for key, value in diagnostics.items():

            if "head_temperatures" in key:

                values.extend(value)

        if values:

            summary[
                "mean_temperature"
            ] = float(
                np.mean(values)
            )

    elif attention_name == "PMA_PIECEWISE":

        pos_values = []
        neg_values = []

        for key, value in diagnostics.items():

            if "pma_pos_temperatures" in key:

                pos_values.extend(value)

            elif "pma_neg_temperatures" in key:

                neg_values.extend(value)

        if pos_values and neg_values:

            pos = np.asarray(
                pos_values,
                dtype=np.float64,
            )

            neg = np.asarray(
                neg_values,
                dtype=np.float64,
            )

            summary[
                "mean_pma_pos"
            ] = float(
                np.mean(pos)
            )

            summary[
                "mean_pma_neg"
            ] = float(
                np.mean(neg)
            )

            summary[
                "mean_pma_gap"
            ] = float(
                np.mean(neg - pos)
            )

            summary[
                "mean_pma_ratio"
            ] = float(
                np.mean(neg / pos)
            )

            summary[
                "fraction_pma_pos_lt_neg"
            ] = float(
                np.mean(pos < neg)
            )

    return summary


# ============================================================
# 15. TRAIN + VALIDATION
# ============================================================

def train_and_eval(
    attention_name: str,
    vocab_size: int,
    train_loader: DataLoader,
    valid_loader: DataLoader,
    accelerator: Accelerator,
    epochs: int,
):

    # --------------------------------------------------------
    # Build model
    # --------------------------------------------------------

    model = TransformerLM(
        vocab_size=vocab_size,
        attention_name=attention_name,
    )

    params = count_parameters(
        model
    )

    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    optimizer = optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = (
        optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=epochs,
        )
    )

    criterion = nn.CrossEntropyLoss()

    # --------------------------------------------------------
    # Prepare with Accelerate
    # --------------------------------------------------------

    (
        model,
        optimizer,
        train_loader,
        valid_loader,
        scheduler,
    ) = accelerator.prepare(
        model,
        optimizer,
        train_loader,
        valid_loader,
        scheduler,
    )

    # --------------------------------------------------------
    # GPU memory measurement
    # --------------------------------------------------------

    if torch.cuda.is_available():

        torch.cuda.reset_peak_memory_stats()

    # --------------------------------------------------------
    # Timing starts after preparation.
    # --------------------------------------------------------

    accelerator.wait_for_everyone()

    if torch.cuda.is_available():

        torch.cuda.synchronize()

    start_time = time.perf_counter()

    # ========================================================
    # TRAINING
    # ========================================================

    for epoch in range(epochs):

        model.train()

        for x, y in train_loader:

            with accelerator.accumulate(
                model
            ):

                outputs = model(x)

                loss = criterion(
                    outputs.reshape(
                        -1,
                        vocab_size,
                    ),
                    y.reshape(-1),
                )

                accelerator.backward(
                    loss
                )

                if (
                    accelerator.sync_gradients
                ):

                    accelerator.clip_grad_norm_(
                        model.parameters(),
                        GRAD_CLIP,
                    )

                optimizer.step()

                optimizer.zero_grad(
                    set_to_none=True
                )

        scheduler.step()

    # --------------------------------------------------------
    # Synchronize before timing.
    # --------------------------------------------------------

    accelerator.wait_for_everyone()

    if torch.cuda.is_available():

        torch.cuda.synchronize()

    elapsed = (
        time.perf_counter()
        - start_time
    )

    # --------------------------------------------------------
    # Distributed timing = slowest process.
    # --------------------------------------------------------

    elapsed_tensor = torch.tensor(
        elapsed,
        dtype=torch.float64,
        device=accelerator.device,
    )

    elapsed_tensor = accelerator.reduce(
        elapsed_tensor,
        reduction="max",
    )

    elapsed = elapsed_tensor.item()

    # ========================================================
    # TRAINING THROUGHPUT
    # ========================================================

    train_tokens_local = (
        len(train_loader.dataset)
        * epochs
    )

    train_tokens_tensor = torch.tensor(
        train_tokens_local,
        dtype=torch.long,
        device=accelerator.device,
    )

    train_tokens_tensor = accelerator.reduce(
        train_tokens_tensor,
        reduction="sum",
    )

    train_tokens = train_tokens_tensor.item()

    train_tokens_per_sec = (
        train_tokens / elapsed
        if elapsed > 0
        else float("nan")
    )

    # ========================================================
    # PEAK GPU MEMORY
    # ========================================================

    if torch.cuda.is_available():

        peak_memory_local = (
            torch.cuda.max_memory_allocated()
        )

    else:

        peak_memory_local = 0

    peak_memory_tensor = torch.tensor(
        peak_memory_local,
        dtype=torch.float64,
        device=accelerator.device,
    )

    peak_memory_tensor = accelerator.reduce(
        peak_memory_tensor,
        reduction="max",
    )

    peak_memory_mb = (
        peak_memory_tensor.item()
        / (1024 ** 2)
    )

    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    total_loss = torch.tensor(
        0.0,
        device=accelerator.device,
        dtype=torch.float64,
    )

    total_tokens = torch.tensor(
        0,
        device=accelerator.device,
        dtype=torch.long,
    )

    total_correct = torch.tensor(
        0,
        device=accelerator.device,
        dtype=torch.long,
    )

    # --------------------------------------------------------
    # Validation timing
    # --------------------------------------------------------

    accelerator.wait_for_everyone()

    if torch.cuda.is_available():

        torch.cuda.synchronize()

    val_start = time.perf_counter()

    with torch.no_grad():

        for x, y in valid_loader:

            outputs = model(x)

            loss_sum = F.cross_entropy(
                outputs.reshape(
                    -1,
                    vocab_size,
                ),
                y.reshape(-1),
                reduction="sum",
            )

            predictions = outputs.argmax(
                dim=-1
            )

            correct = (
                predictions == y
            ).sum()

            n_tokens = torch.tensor(
                y.numel(),
                device=accelerator.device,
                dtype=torch.long,
            )

            total_loss += (
                loss_sum.double()
            )

            total_tokens += n_tokens

            total_correct += (
                correct.long()
            )

    if torch.cuda.is_available():

        torch.cuda.synchronize()

    val_elapsed_local = (
        time.perf_counter()
        - val_start
    )

    val_elapsed_tensor = torch.tensor(
        val_elapsed_local,
        dtype=torch.float64,
        device=accelerator.device,
    )

    val_elapsed_tensor = accelerator.reduce(
        val_elapsed_tensor,
        reduction="max",
    )

    val_elapsed = (
        val_elapsed_tensor.item()
    )

    # --------------------------------------------------------
    # Correct distributed reduction.
    # --------------------------------------------------------

    total_loss = accelerator.reduce(
        total_loss,
        reduction="sum",
    )

    total_tokens = accelerator.reduce(
        total_tokens,
        reduction="sum",
    )

    total_correct = accelerator.reduce(
        total_correct,
        reduction="sum",
    )

    avg_val_loss = (
        total_loss.item()
        / total_tokens.item()
    )

    perplexity = math.exp(
        min(avg_val_loss, 20.0)
    )

    token_accuracy = (
        total_correct.item()
        / total_tokens.item()
    )

    val_tokens_per_sec = (
        total_tokens.item()
        / val_elapsed
        if val_elapsed > 0
        else float("nan")
    )

    # --------------------------------------------------------
    # Bits per token
    #
    # Same information as loss/PPL, but useful for reporting.
    # --------------------------------------------------------

    bits_per_token = (
        avg_val_loss
        / math.log(2.0)
    )

    # ========================================================
    # DIAGNOSTICS
    # ========================================================

    unwrapped_model = (
        accelerator.unwrap_model(
            model
        )
    )

    diagnostics = (
        get_attention_parameters(
            unwrapped_model
        )
    )

    temperature_summary = (
        summarize_temperature_diagnostics(
            diagnostics,
            attention_name,
        )
    )

    return {
        "ppl": perplexity,
        "loss": avg_val_loss,
        "token_accuracy": token_accuracy,
        "bits_per_token": bits_per_token,
        "time_sec": elapsed,
        "train_tokens": train_tokens,
        "train_tokens_per_sec": train_tokens_per_sec,
        "validation_tokens": total_tokens.item(),
        "validation_time_sec": val_elapsed,
        "validation_tokens_per_sec": val_tokens_per_sec,
        "peak_gpu_memory_mb": peak_memory_mb,
        "parameters": params,
        "diagnostics": diagnostics,
        "temperature_summary": temperature_summary,
    }


# ============================================================
# 16. SAVE FULL RESULTS
# ============================================================

def save_full_results(
    results,
    path,
):

    if not results:
        return

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    fieldnames = [
        "sequence_length",
        "attention",
        "seed",
        "ppl",
        "loss",
        "token_accuracy",
        "bits_per_token",
        "time_sec",
        "train_tokens",
        "train_tokens_per_sec",
        "validation_tokens",
        "validation_time_sec",
        "validation_tokens_per_sec",
        "peak_gpu_memory_mb",
        "parameters",
        "mean_temperature",
        "mean_pma_pos",
        "mean_pma_neg",
        "mean_pma_gap",
        "mean_pma_ratio",
        "fraction_pma_pos_lt_neg",
    ]

    with open(
        path,
        "w",
        newline="",
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
        )

        writer.writeheader()

        for row in results:

            writer.writerow(
                {
                    "sequence_length":
                        row["sequence_length"],

                    "attention":
                        row["attention"],

                    "seed":
                        row["seed"],

                    "ppl":
                        row["ppl"],

                    "loss":
                        row["loss"],

                    "token_accuracy":
                        row["token_accuracy"],

                    "bits_per_token":
                        row["bits_per_token"],

                    "time_sec":
                        row["time_sec"],

                    "train_tokens":
                        row["train_tokens"],

                    "train_tokens_per_sec":
                        row["train_tokens_per_sec"],

                    "validation_tokens":
                        row["validation_tokens"],

                    "validation_time_sec":
                        row["validation_time_sec"],

                    "validation_tokens_per_sec":
                        row["validation_tokens_per_sec"],

                    "peak_gpu_memory_mb":
                        row["peak_gpu_memory_mb"],

                    "parameters":
                        row["parameters"],

                    "mean_temperature":
                        row["temperature_summary"][
                            "mean_temperature"
                        ],

                    "mean_pma_pos":
                        row["temperature_summary"][
                            "mean_pma_pos"
                        ],

                    "mean_pma_neg":
                        row["temperature_summary"][
                            "mean_pma_neg"
                        ],

                    "mean_pma_gap":
                        row["temperature_summary"][
                            "mean_pma_gap"
                        ],

                    "mean_pma_ratio":
                        row["temperature_summary"][
                            "mean_pma_ratio"
                        ],

                    "fraction_pma_pos_lt_neg":
                        row["temperature_summary"][
                            "fraction_pma_pos_lt_neg"
                        ],
                }
            )


# ============================================================
# 17. SAVE TEMPERATURE DETAILS
# ============================================================

def save_temperature_details(
    results,
    path,
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    rows = []

    for row in results:

        diagnostics = row[
            "diagnostics"
        ]

        for key, values in diagnostics.items():

            for head_idx, value in enumerate(
                values
            ):

                rows.append(
                    {
                        "sequence_length":
                            row["sequence_length"],

                        "attention":
                            row["attention"],

                        "seed":
                            row["seed"],

                        "diagnostic":
                            key,

                        "head":
                            head_idx,

                        "value":
                            value,
                    }
                )

    if not rows:

        return

    fieldnames = [
        "sequence_length",
        "attention",
        "seed",
        "diagnostic",
        "head",
        "value",
    ]

    with open(
        path,
        "w",
        newline="",
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
        )

        writer.writeheader()

        writer.writerows(rows)


# ============================================================
# 18. FINAL SUMMARY CSV
# ============================================================

def save_summary(
    results,
    path,
):

    if not results:

        return

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    fieldnames = [
        "sequence_length",
        "attention",
        "ppl_mean",
        "ppl_std",
        "loss_mean",
        "loss_std",
        "accuracy_mean",
        "accuracy_std",
        "bits_per_token_mean",
        "time_mean_sec",
        "time_std_sec",
        "train_tokens_per_sec_mean",
        "validation_tokens_per_sec_mean",
        "peak_gpu_memory_mean_mb",
        "parameters",
    ]

    with open(
        path,
        "w",
        newline="",
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
        )

        writer.writeheader()

        for seq_len in SEQ_LENS:

            for name in ATTENTION_MODELS:

                rows = [
                    r
                    for r in results
                    if r["sequence_length"] == seq_len
                    and r["attention"] == name
                ]

                if not rows:

                    continue

                ppls = np.asarray(
                    [
                        r["ppl"]
                        for r in rows
                    ],
                    dtype=np.float64,
                )

                losses = np.asarray(
                    [
                        r["loss"]
                        for r in rows
                    ],
                    dtype=np.float64,
                )

                accuracies = np.asarray(
                    [
                        r["token_accuracy"]
                        for r in rows
                    ],
                    dtype=np.float64,
                )

                bpt = np.asarray(
                    [
                        r["bits_per_token"]
                        for r in rows
                    ],
                    dtype=np.float64,
                )

                times = np.asarray(
                    [
                        r["time_sec"]
                        for r in rows
                    ],
                    dtype=np.float64,
                )

                train_tps = np.asarray(
                    [
                        r["train_tokens_per_sec"]
                        for r in rows
                    ],
                    dtype=np.float64,
                )

                val_tps = np.asarray(
                    [
                        r["validation_tokens_per_sec"]
                        for r in rows
                    ],
                    dtype=np.float64,
                )

                memory = np.asarray(
                    [
                        r["peak_gpu_memory_mb"]
                        for r in rows
                    ],
                    dtype=np.float64,
                )

                writer.writerow(
                    {
                        "sequence_length":
                            seq_len,

                        "attention":
                            name,

                        "ppl_mean":
                            np.mean(ppls),

                        "ppl_std":
                            np.std(
                                ppls,
                                ddof=1,
                            ),

                        "loss_mean":
                            np.mean(losses),

                        "loss_std":
                            np.std(
                                losses,
                                ddof=1,
                            ),

                        "accuracy_mean":
                            np.mean(
                                accuracies
                            ),

                        "accuracy_std":
                            np.std(
                                accuracies,
                                ddof=1,
                            ),

                        "bits_per_token_mean":
                            np.mean(bpt),

                        "time_mean_sec":
                            np.mean(times),

                        "time_std_sec":
                            np.std(
                                times,
                                ddof=1,
                            ),

                        "train_tokens_per_sec_mean":
                            np.mean(train_tps),

                        "validation_tokens_per_sec_mean":
                            np.mean(val_tps),

                        "peak_gpu_memory_mean_mb":
                            np.mean(memory),

                        "parameters":
                            rows[0]["parameters"],
                    }
                )


# ============================================================
# 19. PAIRED STATISTICS
#
# IMPORTANT:
# Each method is compared against MHSA using the SAME seeds.
# This is the appropriate paired design for this benchmark.
# ============================================================

def print_statistical_analysis(
    results,
    seq_len,
):

    print()
    print("=" * 110)

    print(
        f"STATISTICAL ANALYSIS VS MHSA "
        f"| SEQ_LEN={seq_len}"
    )

    print("=" * 110)

    mhsa_rows = [
        r
        for r in results
        if r["sequence_length"] == seq_len
        and r["attention"] == "MHSA"
    ]

    mhsa_by_seed = {
        r["seed"]: r
        for r in mhsa_rows
    }

    for name in [
        "HeadTemp",
        "PMA_PIECEWISE",
    ]:

        method_rows = [
            r
            for r in results
            if r["sequence_length"] == seq_len
            and r["attention"] == name
        ]

        method_by_seed = {
            r["seed"]: r
            for r in method_rows
        }

        common_seeds = sorted(
            set(mhsa_by_seed)
            & set(method_by_seed)
        )

        if len(common_seeds) < 2:

            print(
                f"\n{name}: insufficient paired seeds."
            )

            continue

        mhsa_ppl = np.asarray(
            [
                mhsa_by_seed[s]["ppl"]
                for s in common_seeds
            ],
            dtype=np.float64,
        )

        method_ppl = np.asarray(
            [
                method_by_seed[s]["ppl"]
                for s in common_seeds
            ],
            dtype=np.float64,
        )

        mhsa_loss = np.asarray(
            [
                mhsa_by_seed[s]["loss"]
                for s in common_seeds
            ],
            dtype=np.float64,
        )

        method_loss = np.asarray(
            [
                method_by_seed[s]["loss"]
                for s in common_seeds
            ],
            dtype=np.float64,
        )

        # ----------------------------------------------------
        # Paired PPL test
        # ----------------------------------------------------

        ppl_diff = (
            method_ppl
            - mhsa_ppl
        )

        t_ppl, p_ppl = (
            stats.ttest_rel(
                method_ppl,
                mhsa_ppl,
            )
        )

        # ----------------------------------------------------
        # Paired loss test
        # ----------------------------------------------------

        loss_diff = (
            method_loss
            - mhsa_loss
        )

        t_loss, p_loss = (
            stats.ttest_rel(
                method_loss,
                mhsa_loss,
            )
        )

        print()
        print(
            f"{name} vs MHSA"
        )

        print(
            f"  Seeds             : "
            f"{common_seeds}"
        )

        print(
            f"  Mean PPL diff     : "
            f"{np.mean(ppl_diff):+.6f}"
        )

        print(
            f"  PPL SD of diff    : "
            f"{np.std(ppl_diff, ddof=1):.6f}"
        )

        print(
            f"  PPL paired t      : "
            f"{t_ppl:.6f}"
        )

        print(
            f"  PPL p-value       : "
            f"{p_ppl:.6f}"
        )

        print(
            f"  Mean loss diff    : "
            f"{np.mean(loss_diff):+.8f}"
        )

        print(
            f"  Loss SD of diff   : "
            f"{np.std(loss_diff, ddof=1):.8f}"
        )

        print(
            f"  Loss paired t     : "
            f"{t_loss:.6f}"
        )

        print(
            f"  Loss p-value      : "
            f"{p_loss:.6f}"
        )

        if p_ppl < 0.05:

            if np.mean(ppl_diff) < 0:

                print(
                    "  PPL conclusion    : "
                    "STATISTICALLY SIGNIFICANT IMPROVEMENT"
                )

            else:

                print(
                    "  PPL conclusion    : "
                    "STATISTICALLY SIGNIFICANT REGRESSION"
                )

        else:

            print(
                "  PPL conclusion    : "
                "NOT STATISTICALLY SIGNIFICANT"
            )

        if p_loss < 0.05:

            if np.mean(loss_diff) < 0:

                print(
                    "  Loss conclusion   : "
                    "STATISTICALLY SIGNIFICANT IMPROVEMENT"
                )

            else:

                print(
                    "  Loss conclusion   : "
                    "STATISTICALLY SIGNIFICANT REGRESSION"
                )

        else:

            print(
                "  Loss conclusion   : "
                "NOT STATISTICALLY SIGNIFICANT"
            )

    print("=" * 110)


# ============================================================
# 20. MAIN BENCHMARK
# ============================================================

def run_nlp_benchmark():

    accelerator = Accelerator(
        mixed_precision="fp16",
        gradient_accumulation_steps=(
            GRADIENT_ACCUMULATION_STEPS
        ),
    )

    # --------------------------------------------------------
    # Environment
    # --------------------------------------------------------

    if accelerator.is_main_process:

        print()
        print("=" * 110)
        print(
            "ATTENTION MECHANISM WIKITEXT "
            "MULTI-SEQUENCE-LENGTH BENCHMARK"
        )
        print("=" * 110)

        print(
            f"PyTorch              : "
            f"{torch.__version__}"
        )

        print(
            f"CUDA available       : "
            f"{torch.cuda.is_available()}"
        )

        if torch.cuda.is_available():

            print(
                f"CUDA devices         : "
                f"{torch.cuda.device_count()}"
            )

        print(
            f"Accelerate device    : "
            f"{accelerator.device}"
        )

        print(
            f"Processes            : "
            f"{accelerator.num_processes}"
        )

        print(
            f"Attention mechanisms : "
            f"{ATTENTION_MODELS}"
        )

        print(
            f"Seeds                : "
            f"{SEEDS}"
        )

        print(
            f"Sequence lengths     : "
            f"{SEQ_LENS}"
        )

        print(
            f"Epochs               : "
            f"{EPOCHS}"
        )

        print(
            f"Train batch size     : "
            f"{TRAIN_BATCH_SIZE}"
        )

        print(
            f"Valid batch size     : "
            f"{VALID_BATCH_SIZE}"
        )

        print(
            f"Gradient accumulation: "
            f"{GRADIENT_ACCUMULATION_STEPS}"
        )

        print(
            f"d_model             : "
            f"{D_MODEL}"
        )

        print(
            f"Heads               : "
            f"{N_HEADS}"
        )

        print(
            f"Layers              : "
            f"{N_LAYERS}"
        )

        print(
            f"FFN                 : "
            f"{DIM_FEEDFORWARD}"
        )

        print(
            f"Dropout             : "
            f"{DROPOUT}"
        )

        print(
            f"Maximum position    : "
            f"{MAX_SEQ_LEN}"
        )

        print(
            f"PMA T+ initialization: "
            f"{PMA_POS_TEMP_INIT}"
        )

        print(
            f"PMA T- initialization: "
            f"{PMA_NEG_TEMP_INIT}"
        )

        print("=" * 110)

    # --------------------------------------------------------
    # Results
    # --------------------------------------------------------

    all_results = []

    total_jobs = (
        len(SEQ_LENS)
        * len(ATTENTION_MODELS)
        * len(SEEDS)
    )

    job = 0

    # ========================================================
    # SEQUENCE LENGTH LOOP
    # ========================================================

    for seq_len in SEQ_LENS:

        if accelerator.is_main_process:

            print()
            print("#" * 110)

            print(
                f"SEQUENCE LENGTH: "
                f"{seq_len}"
            )

            print("#" * 110)

        # ----------------------------------------------------
        # Create datasets specifically for this sequence length.
        # ----------------------------------------------------

        (
            train_loader,
            valid_loader,
            vocab_size,
        ) = get_nlp_dataloaders(
            accelerator=accelerator,
            seq_len=seq_len,
        )

        if accelerator.is_main_process:

            print(
                f"\nSequence length: "
                f"{seq_len}"
            )

            print(
                f"Vocabulary size: "
                f"{vocab_size:,}"
            )

            print(
                f"Train batches: "
                f"{len(train_loader):,}"
            )

            print(
                f"Validation batches: "
                f"{len(valid_loader):,}"
            )

        # ====================================================
        # ATTENTION METHODS
        # ====================================================

        for attention_name in ATTENTION_MODELS:

            if accelerator.is_main_process:

                print()
                print(
                    "-" * 100
                )

                print(
                    f"ATTENTION: "
                    f"{attention_name} "
                    f"| SEQ_LEN={seq_len}"
                )

                print(
                    "-" * 100
                )

            # =================================================
            # SEEDS
            # =================================================

            for seed in SEEDS:

                job += 1

                # ------------------------------------------------
                # Same seed for corresponding experiments.
                # ------------------------------------------------

                set_seed(seed)

                if accelerator.is_main_process:

                    print()
                    print(
                        f"JOB "
                        f"{job}/{total_jobs} | "
                        f"SEQ={seq_len} | "
                        f"{attention_name} | "
                        f"Seed={seed}"
                    )

                # ------------------------------------------------
                # Train
                # ------------------------------------------------

                metrics = train_and_eval(
                    attention_name=attention_name,
                    vocab_size=vocab_size,
                    train_loader=train_loader,
                    valid_loader=valid_loader,
                    accelerator=accelerator,
                    epochs=EPOCHS,
                )

                result = {
                    "sequence_length":
                        seq_len,

                    "attention":
                        attention_name,

                    "seed":
                        seed,

                    **metrics,
                }

                all_results.append(
                    result
                )

                # ------------------------------------------------
                # Print result
                # ------------------------------------------------

                if accelerator.is_main_process:

                    print()
                    print(
                        f"RESULT | "
                        f"{attention_name} | "
                        f"Seq={seq_len} | "
                        f"Seed={seed}"
                    )

                    print(
                        f"  PPL                  : "
                        f"{metrics['ppl']:.4f}"
                    )

                    print(
                        f"  Loss                 : "
                        f"{metrics['loss']:.6f}"
                    )

                    print(
                        f"  Token accuracy       : "
                        f"{metrics['token_accuracy']:.6f}"
                    )

                    print(
                        f"  Bits/token           : "
                        f"{metrics['bits_per_token']:.6f}"
                    )

                    print(
                        f"  Training time        : "
                        f"{metrics['time_sec']:.1f}s"
                    )

                    print(
                        f"  Training tokens/sec  : "
                        f"{metrics['train_tokens_per_sec']:.1f}"
                    )

                    print(
                        f"  Validation tokens/sec: "
                        f"{metrics['validation_tokens_per_sec']:.1f}"
                    )

                    print(
                        f"  Peak GPU memory      : "
                        f"{metrics['peak_gpu_memory_mb']:.1f} MB"
                    )

                    print(
                        f"  Parameters           : "
                        f"{metrics['parameters']:,}"
                    )

                    # ------------------------------------------------
                    # Temperature diagnostics
                    # ------------------------------------------------

                    for key, value in (
                        metrics["diagnostics"].items()
                    ):

                        print(
                            f"  {key}: "
                            f"{np.round(value, 4).tolist()}"
                        )

                    temp = (
                        metrics[
                            "temperature_summary"
                        ]
                    )

                    if attention_name == "HeadTemp":

                        print(
                            f"  Mean HeadTemp T      : "
                            f"{temp['mean_temperature']:.6f}"
                        )

                    if (
                        attention_name
                        == "PMA_PIECEWISE"
                    ):

                        print(
                            f"  Mean PMA T+          : "
                            f"{temp['mean_pma_pos']:.6f}"
                        )

                        print(
                            f"  Mean PMA T-          : "
                            f"{temp['mean_pma_neg']:.6f}"
                        )

                        print(
                            f"  Mean T- - T+         : "
                            f"{temp['mean_pma_gap']:.6f}"
                        )

                        print(
                            f"  Mean T-/T+           : "
                            f"{temp['mean_pma_ratio']:.6f}"
                        )

                        print(
                            f"  Fraction T+ < T-     : "
                            f"{temp['fraction_pma_pos_lt_neg']:.4f}"
                        )

                    # ------------------------------------------------
                    # Incremental saves
                    # ------------------------------------------------

                    save_full_results(
                        all_results,
                        OUTPUT_DIR
                        / "full_results.csv",
                    )

                    save_temperature_details(
                        all_results,
                        OUTPUT_DIR
                        / "temperature_details.csv",
                    )

                accelerator.wait_for_everyone()

                # ------------------------------------------------
                # Cleanup
                # ------------------------------------------------

                torch.cuda.empty_cache()

        # ====================================================
        # SEQUENCE-SPECIFIC STATISTICS
        # ====================================================

        if accelerator.is_main_process:

            print_statistical_analysis(
                all_results,
                seq_len,
            )

    # ========================================================
    # FINAL SUMMARY
    # ========================================================

    if accelerator.is_main_process:

        # ----------------------------------------------------
        # Save final files
        # ----------------------------------------------------

        save_full_results(
            all_results,
            OUTPUT_DIR
            / "full_results.csv",
        )

        save_temperature_details(
            all_results,
            OUTPUT_DIR
            / "temperature_details.csv",
        )

        save_summary(
            all_results,
            OUTPUT_DIR
            / "summary_results.csv",
        )

        # ====================================================
        # FINAL SUMMARY PRINT
        # ====================================================

        print()
        print("=" * 120)

        print(
            "FINAL MULTI-SEQUENCE-LENGTH "
            "ATTENTION BENCHMARK"
        )

        print("=" * 120)

        for seq_len in SEQ_LENS:

            print()
            print(
                "=" * 110
            )

            print(
                f"SEQUENCE LENGTH = "
                f"{seq_len}"
            )

            print(
                "=" * 110
            )

            print(
                f"{'Attention':<20}"
                f"{'PPL':>18}"
                f"{'Loss':>18}"
                f"{'Acc':>15}"
                f"{'Time(s)':>18}"
                f"{'Tok/s':>18}"
            )

            print(
                "-" * 110
            )

            for name in ATTENTION_MODELS:

                rows = [
                    r
                    for r in all_results
                    if r["sequence_length"] == seq_len
                    and r["attention"] == name
                ]

                if not rows:

                    continue

                ppls = np.asarray(
                    [
                        r["ppl"]
                        for r in rows
                    ],
                    dtype=np.float64,
                )

                losses = np.asarray(
                    [
                        r["loss"]
                        for r in rows
                    ],
                    dtype=np.float64,
                )

                accuracies = np.asarray(
                    [
                        r["token_accuracy"]
                        for r in rows
                    ],
                    dtype=np.float64,
                )

                times = np.asarray(
                    [
                        r["time_sec"]
                        for r in rows
                    ],
                    dtype=np.float64,
                )

                tps = np.asarray(
                    [
                        r["train_tokens_per_sec"]
                        for r in rows
                    ],
                    dtype=np.float64,
                )

                print(
                    f"{name:<20}"
                    f"{ppls.mean():>9.4f} "
                    f"+/- {ppls.std(ddof=1):<6.4f}"
                    f"{losses.mean():>9.6f} "
                    f"+/- {losses.std(ddof=1):<6.6f}"
                    f"{accuracies.mean():>8.4f} "
                    f"+/- {accuracies.std(ddof=1):<5.4f}"
                    f"{times.mean():>9.1f} "
                    f"+/- {times.std(ddof=1):<5.1f}"
                    f"{tps.mean():>10.1f}"
                )

            print(
                "=" * 110
            )

        # ====================================================
        # CROSS-LENGTH PPL RANKING
        # ====================================================

        print()
        print("=" * 110)

        print(
            "PPL RANKING BY SEQUENCE LENGTH"
        )

        print("=" * 110)

        for seq_len in SEQ_LENS:

            ranking_values = []

            for name in ATTENTION_MODELS:

                rows = [
                    r
                    for r in all_results
                    if r["sequence_length"] == seq_len
                    and r["attention"] == name
                ]

                if rows:

                    mean_ppl = np.mean(
                        [
                            r["ppl"]
                            for r in rows
                        ]
                    )

                    ranking_values.append(
                        (
                            name,
                            mean_ppl,
                        )
                    )

            ranking_values.sort(
                key=lambda x: x[1]
            )

            print()
            print(
                f"Seq={seq_len}"
            )

            for rank, (
                name,
                mean_ppl,
            ) in enumerate(
                ranking_values,
                start=1,
            ):

                print(
                    f"  {rank}. "
                    f"{name:<20}"
                    f"PPL={mean_ppl:.4f}"
                )

        print()
        print("=" * 110)

        # ====================================================
        # FINAL FILE LOCATIONS
        # ====================================================

        print()
        print(
            "RESULT FILES"
        )

        print(
            f"  Full results       : "
            f"{OUTPUT_DIR / 'full_results.csv'}"
        )

        print(
            f"  Summary results    : "
            f"{OUTPUT_DIR / 'summary_results.csv'}"
        )

        print(
            f"  Temperature details: "
            f"{OUTPUT_DIR / 'temperature_details.csv'}"
        )

        print()
        print(
            "Benchmark completed."
        )


# ============================================================
# ENTRY POINT
# ============================================================

if __name__ == "__main__":

    run_nlp_benchmark()

In [ ]:
!torchrun --nproc_per_node=2 /kaggle/working/benchmark.py